# Task 1.1 — Core Contribution / Architecture
**Paper:** DynaMMo: Mining and Summarization of Coevolving Sequences with Missing Values  
**Authors:** Li, McCann, Pollard, Faloutsos  
**Venue:** KDD 2009  
**Roll Number:** [YOUR ROLL NUMBER]


## Step-by-Step Method Description

---

### Step 1: Input Representation
- **Description:** The input is a collection of `d` co-evolving time series, each of length `T`, organised as a matrix `X ∈ ℝ^{T×d}`. Some entries `X[t,j]` may be missing (NaN). Missing values arise from sensor dropout, packet loss, or occlusion — the motivating real-world problem of the paper.
- **Reference:** Section 1 (Introduction) and Section 2 (Problem Definition), Figure 1.
- **Purpose:** To frame the problem as recovery of a structured signal from incomplete observations, rather than as a simple interpolation task.

---

### Step 2: Linear Dynamical System (LDS) Model Definition
- **Description:** The paper models the `d` observed sequences as being driven by a lower-dimensional latent state `z_t ∈ ℝ^k` (where `k ≪ d`). Two equations define the model:
  - **State transition:** `z_t = A·z_{t-1} + ε_t`,  `ε_t ~ N(0, Q)`
  - **Observation emission:** `x_t = C·z_t + δ_t`,   `δ_t ~ N(0, R)`
  The transition matrix `A` captures autoregressive structure (how past hidden states predict future ones), and `C` maps hidden states to observed sequences.
- **Reference:** Section 3, Equations (1) and (2).
- **Purpose:** This generative model allows the paper to exploit shared latent structure across all `d` sequences simultaneously — something pointwise interpolation cannot do. If all sequences share a common underlying driver, the model can recover missing values even in a sequence that is entirely unobserved at some timestep, using information from other sequences.

---

### Step 3: Kalman Filter — Forward Inference Pass
- **Description:** Given parameters `(A, C, Q, R)`, the Kalman Filter runs forward through time `t = 1, …, T`. At each step it:
  1. **Predicts:** `μ̂_{t|t-1} = A·μ_{t-1}`,  `V̂_{t|t-1} = A·V_{t-1}·A^T + Q`
  2. **Updates:** uses only the observed dimensions of `x_t` to correct the prediction via the Kalman gain `K = V̂·C_obs^T·(C_obs·V̂·C_obs^T + R_obs)^{-1}`
  When `x_t` is fully missing, the update step is skipped — this is the paper's key handling of missing values.
- **Reference:** Section 3.1, Algorithm 1 (Filter step), Equations (3)–(7).
- **Purpose:** Computes `P(z_t | x_{1:t})` — the best estimate of the latent state given all observations up to time `t`. This is the forward pass of inference.

---

### Step 4: RTS Smoother — Backward Inference Pass
- **Description:** After the forward Kalman Filter, the RTS (Rauch-Tung-Striebel) Smoother runs backward from `t = T` to `t = 1`. The smoother gain is `L_t = V_t·A^T·V̂_{t+1|t}^{-1}`. It refines each state estimate using future observations:
  `μ_t^s = μ_t + L_t·(μ_{t+1}^s - μ̂_{t+1|t})`
- **Reference:** Section 3.1, Algorithm 1 (Smoother step), Equations (8)–(10).
- **Purpose:** Computes `P(z_t | x_{1:T})` — uses the entire sequence (both past and future) to estimate each hidden state. This is critical for accurate imputation: a missing value at time `t` benefits from observations at `t+100` if they share the same latent driver.

---

### Step 5: EM Algorithm — Parameter Learning (M-step)
- **Description:** Parameters `(A, C, Q, R)` are unknown and are learned by maximising the log-likelihood `log P(X | A, C, Q, R)` via Expectation-Maximisation. The E-step is the Kalman Filter + Smoother above. The M-step uses closed-form updates derived from the sufficient statistics `{E[z_t z_t^T], E[z_t z_{t-1}^T], E[x_t z_t^T]}`:
  - `A_new = [Σ z_t z_{t-1}^T][Σ z_{t-1} z_{t-1}^T]^{-1}`
  - `C_new = [Σ x_t z_t^T][Σ z_t z_t^T]^{-1}`
  - `Q_new = (1/T-1)[Σ z_t z_t^T - A·Σ z_{t-1} z_t^T]`
  - `R_new = diag((1/T)[Σ x_t x_t^T - C·Σ x_t z_t^T])`
- **Reference:** Section 3.1, Equations (11)–(16).
- **Purpose:** Iterates until convergence to find the maximum-likelihood LDS parameters that best explain the observed (partial) data.

---

### Step 6: Missing Value Imputation (Key Contribution)
- **Description:** Once the model is trained and smoothed states `μ_t^s` are computed, the imputed value for any missing dimension `j` at time `t` is:  
  `x̂_{t,j} = [C · μ_t^s]_j`  
  The smoother uses all available observations (including from other sequences at the same timestep) to estimate `z_t`, which is then projected back into observation space via `C`.
- **Reference:** Section 3.2, Equation (17).
- **Purpose:** This is the primary novel contribution — recovering missing values by leveraging the shared latent dynamics across all co-evolving sequences.

---

### Step 7: Compression and Summarisation (Secondary Contribution)
- **Description:** The full dataset `X ∈ ℝ^{T×d}` can be stored compactly as the latent state sequence `{z_t} ∈ ℝ^{T×k}` plus parameters `(A, C, Q, R)`. The compression ratio is:
  `CR = (T·d) / (T·k + k·d + 2k² + d)`
- **Reference:** Section 4.2.
- **Purpose:** When `k ≪ d`, the LDS provides a compact, low-rank summary of the entire dataset while still enabling approximate reconstruction of all sequences.

---

## Final Summary Sentence

DynaMMo solves the problem of recovering missing values in multiple co-evolving time series by fitting a shared Linear Dynamical System via EM — the authors claim this outperforms naive imputation methods (mean fill, linear interpolation) because it captures the **joint temporal and cross-sequence structure** that drives all observed signals simultaneously, enabling recovery of missing values even when entire sequences are unobserved at a timestep.
